<a href="https://colab.research.google.com/github/harsh22201/CSE643-Artificial-Intelligence/blob/main/Assignment%202/Visulaization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import networkx as nx
from collections import defaultdict, deque

## ****IMPORTANT****
## Don't import or use any other libraries other than defined above
## Otherwise your code file will be rejected in the automated testing

# ------------------ Global Variables ------------------
route_to_stops = defaultdict(list)  # Mapping of route IDs to lists of stops
trip_to_route = {}                   # Mapping of trip IDs to route IDs
stop_trip_count = defaultdict(int)    # Count of trips for each stop
fare_rules = {}                      # Mapping of route IDs to fare information
merged_fare_df = None                # To be initialized in create_kb()

# Load static data from GTFS (General Transit Feed Specification) files
df_stops = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GTFS/stops.txt')
df_routes = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GTFS/stops.txt')
df_stop_times = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GTFS/stop_times.txt')
df_fare_attributes = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GTFS/fare_attributes.txt')
df_trips = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GTFS/trips.txt')
df_fare_rules = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/GTFS/fare_rules.txt')

# ------------------ Function Definitions ------------------

# Function to create knowledge base from the loaded data
def create_kb():
    """
    Create knowledge base by populating global variables with information from loaded datasets.
    It establishes the relationships between routes, trips, stops, and fare rules.

    Returns:
        None
    """
    global route_to_stops, trip_to_route, stop_trip_count, fare_rules, merged_fare_df

    # Create trip_id to route_id mapping
    trip_to_route = dict(zip(df_trips["trip_id"], df_trips["route_id"]))

    # Map route_id to a list of stops in order of their sequence
    for trip_id, stop_id, stop_sequence in zip(df_stop_times["trip_id"], df_stop_times["stop_id"],df_stop_times["stop_sequence"]):
        route_id = trip_to_route[trip_id]
        if(stop_id not in route_to_stops[route_id]):
            route_to_stops[route_id].append(stop_id)

    # Ensure each route only has unique stops
    # for route_id in route_to_stops:
    #     route_to_stops[route_id] = list(set(route_to_stops[route_id]))

    # Count trips per stop
    stop_trip_count.update(dict(df_stop_times["stop_id"].value_counts()))

    # Create fare rules for routes

    # Merge fare rules and attributes into a single DataFrame
    merged_fare_df = pd.merge(df_fare_attributes, df_fare_rules, on="fare_id", how='inner')

In [ ]:
create_kb()
print("done")

done


In [ ]:

# Visualize the stop-route graph interactively
def visualize_stop_route_graph_interactive(route_to_stops):
    """
    Visualize the stop-route graph using Plotly for interactive exploration,
    with nodes and edges colored using a color scale for a large number of routes.

    Args:
        route_to_stops (dict): A dictionary mapping route IDs to lists of stops.

    Returns:
        None
    """
    # Initialize directed graph
    G = nx.DiGraph()

    # Generate color scale based on route ID range
    route_ids = list(route_to_stops.keys())
    colorscale = 'Viridis'  # Can be changed to any Plotly color scale

    # Normalize route IDs to [0, 1] for color mapping
    route_id_normalized = {route_id: i / (len(route_ids) - 1) for i, route_id in enumerate(route_ids)}

    # Build graph from route_to_stops data and assign colors
    node_colors = {}
    edge_text_x = []
    edge_text_y = []
    edge_text_labels = []  # Store route IDs for edges
    for route_id, route_stops in route_to_stops.items():
        color_value = route_id_normalized[route_id]
        for i in range(len(route_stops) - 1):
            G.add_edge(route_stops[i], route_stops[i + 1], route=route_id)
            node_colors[route_stops[i]] = color_value
            node_colors[route_stops[i + 1]] = color_value

    # Use kamada_kawai_layout for a balanced layout
    pos = nx.kamada_kawai_layout(G)

    # Prepare edge and node coordinates for plotting
    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

        # Calculate and store midpoint for text labels
        edge_text_x.append((x0 + x1) / 2)
        edge_text_y.append((y0 + y1) / 2)
        edge_text_labels.append(f"Route: {G.edges[edge]['route']}")

    node_x = []
    node_y = []
    node_color_list = []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_color_list.append(node_colors[node])

    # Define edge trace without hover text
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        mode='lines',
        line=dict(width=0.5),
        hoverinfo='none'
    )

    # Define node trace with color scale
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        marker=dict(
            size=5,
            color=node_color_list,
            colorscale=colorscale,
            line=dict(width=0.5)),
        text=list(G.nodes),
        textposition="top center",
        hoverinfo='text'
    )

    # Define edge text trace for route IDs
    edge_text_trace = go.Scatter(
        x=edge_text_x, y=edge_text_y,
        mode='text',
        text=edge_text_labels,
        textposition="middle center",
        textfont=dict(size=5),  # Set font size here
        hoverinfo='text'
    )

    # Create figure and set layout
    fig = go.Figure(data=[edge_trace, node_trace, edge_text_trace],
                    layout=go.Layout(
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20, l=5, r=5, t=40),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        width = 1500,
                        height = 1000
                        ))


    # Display the interactive plot
    fig.show()



In [ ]:
visualize_stop_route_graph_interactive(route_to_stops)
